In [3]:
import numpy as np

rng = np.random.default_rng(123)
n = 200
size = rng.normal(1500, 400, n)
beds = rng.integers(1, 5, n)
age = np.clip(rng.normal(20, 10, n), 0, None)
price = 50_000 + 120 * size + 15_000 * beds - 1_000 * age
price += rng.normal(0, 25_000, n)
X = np.column_stack([size, beds, age]).astype(float)
y = price.astype(float).reshape(-1, 1)

idx = rng.permutation(n)
split = int(0.8 * n)
tr_idx, va_idx = idx[:split], idx[split:]
Xtr, Xva = X[tr_idx], X[va_idx]
ytr, yva = y[tr_idx], y[va_idx]

mu, sigma = Xtr.mean(axis=0, keepdims=True), Xtr.std(axis=0, keepdims=True) + 1e-8
Xtr_s, Xva_s = (Xtr - mu) / sigma, (Xva - mu) / sigma

inp, hid, out = 3, 8, 1
W1 = rng.normal(0, 0.01, (inp, hid))
b1 = np.zeros((1, hid))
W2 = rng.normal(0, 0.01, (hid, out))
b2 = np.zeros((1, out))

def relu(z): return np.maximum(0, z)
def relu_deriv(a): return (a > 0).astype(a.dtype)
def forward(Xs):
    z1 = Xs @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    return z1, a1, z2
def mse(y_true, y_pred): return np.mean((y_true - y_pred)**2)
def rmse(y_true, y_pred): return np.sqrt(mse(y_true, y_pred))
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1.0 - ss_res / (ss_tot + 1e-12)

lr = 0.005
epochs = 5000
patience = 200
best_val = np.inf
wait = 0

for ep in range(1, epochs + 1):
    z1, a1, yhat_tr = forward(Xtr_s)
    loss = mse(ytr, yhat_tr)
    
    dL_dy = (yhat_tr - ytr) * (2.0 / ytr.shape[0])
    dW2 = a1.T @ dL_dy
    db2 = np.sum(dL_dy, axis=0, keepdims=True)
    da1 = dL_dy @ W2.T
    dz1 = da1 * relu_deriv(a1)
    dW1 = Xtr_s.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)
    
    W2 -= lr * dW2; b2 -= lr * db2
    W1 -= lr * dW1; b1 -= lr * db1
    
    _, _, yhat_va = forward(Xva_s)
    val_rmse = rmse(yva, yhat_va)
    
    if val_rmse + 1e-6 < best_val:
        best_val = val_rmse
        wait = 0
        best_params = (W1.copy(), b1.copy(), W2.copy(), b2.copy())
    else:
        wait += 1
    if wait >= patience:
        break

W1, b1, W2, b2 = best_params
_, _, yhat_tr = forward(Xtr_s)
_, _, yhat_va = forward(Xva_s)

print(f"Train RMSE: {rmse(ytr, yhat_tr):.2f}, R2: {r2(ytr, yhat_tr):.3f}")
print(f"Valid RMSE: {rmse(yva, yhat_va):.2f}, R2: {r2(yva, yhat_va):.3f}")

for i in range(min(5, Xva.shape[0])):
    print(f"Features size={Xva[i,0]:.0f} sqft, beds={int(Xva[i,1])}, age={Xva[i,2]:.0f} "
          f"| true={yva[i,0]:.0f} | pred={yhat_va[i,0]:.0f}")


Train RMSE: 252786.48, R2: -20.261
Valid RMSE: 256453.84, R2: -23.460
Features size=1585 sqft, beds=4, age=21 | true=302408 | pred=4031
Features size=1243 sqft, beds=4, age=4 | true=220295 | pred=3563
Features size=1245 sqft, beds=4, age=7 | true=250548 | pred=3676
Features size=1442 sqft, beds=1, age=0 | true=267149 | pred=2582
Features size=1676 sqft, beds=3, age=31 | true=301395 | pred=3882


C:\Users\siddk\AppData\Local\Temp\ipykernel_29756\3027017225.py:35: RuntimeWarning: overflow encountered in square
  def mse(y_true, y_pred): return np.mean((y_true - y_pred)**2)
C:\Users\siddk\AppData\Local\Temp\ipykernel_29756\3027017225.py:33: RuntimeWarning: overflow encountered in matmul
  z2 = a1 @ W2 + b2
C:\Users\siddk\AppData\Local\Temp\ipykernel_29756\3027017225.py:53: RuntimeWarning: invalid value encountered in matmul
  dW2 = a1.T @ dL_dy
C:\Users\siddk\AppData\Local\Temp\ipykernel_29756\3027017225.py:55: RuntimeWarning: overflow encountered in matmul
  da1 = dL_dy @ W2.T
C:\Users\siddk\AppData\Local\Temp\ipykernel_29756\3027017225.py:56: RuntimeWarning: invalid value encountered in multiply
  dz1 = da1 * relu_deriv(a1)
